
# ⚙️ Desafio 1: Validando dados da camada Bronze
Na Aula 2, organizamos a estrutura do projeto e realizamos a ingestão dos dados para a camada Bronze.

Nesta etapa, nossa preocupação é preservar os dados próximos de como chegaram na fonte e garantir que eles estejam disponíveis para as próximas etapas do pipeline.

Seu desafio é usar o Genie Code para ajudar você a criar uma consulta que permita verificar os dados que acabaram de ser ingeridos.

A consulta deve responder a três perguntas simples:

Quantos registros existem na tabela?

Quais são as colunas disponíveis?

Quais são os primeiros registros armazenados?

O objetivo não é modificar os dados. Queremos apenas inspecionar e validar a ingestão realizada durante a aula.

✍️ A ideia aqui é praticar algo comum no dia a dia de Engenharia de Dados: depois de executar uma etapa do pipeline, precisamos verificar se o resultado corresponde ao que esperávamos.

## GENIE - PROMPT CRIADO PARA AUXILIAR O DESAFIO

“Tenho uma tabela na camada Bronze de um pipeline de Engenharia de Dados no Databricks. Os dados já foram ingeridos e quero apenas validar o resultado, sem modificar a tabela.”

Depois, seja específico sobre o que deseja verificar: 

quantidade de registros; 

estrutura da tabela; 

primeiros registros; 

nenhuma alteração nos dados.

Você pode realizar a exploração com SQL ou PySpark. Escolha a abordagem com a qual se sentir mais confortável.

In [0]:
%sql --name bronze_resumo
-- Resumo da camada Bronze: tabelas ingeridas e total de registros
SELECT 'vra'                   AS tabela, COUNT(*) AS total_registros FROM voebem.bronze.vra
UNION ALL SELECT 'aerodromos',            COUNT(*) FROM voebem.bronze.aerodromos
UNION ALL SELECT 'empresas_nacionais',     COUNT(*) FROM voebem.bronze.empresas_nacionais
UNION ALL SELECT 'empresas_estrangeiras',  COUNT(*) FROM voebem.bronze.empresas_estrangeiras
UNION ALL SELECT 'codigos_operacao',       COUNT(*) FROM voebem.bronze.codigos_operacao
ORDER BY total_registros DESC

In [0]:
%sql
-- Estrutura de colunas de cada tabela na camada Bronze
SELECT table_name        AS tabela,
       column_name       AS coluna,
       data_type         AS tipo,
       ordinal_position  AS posicao
FROM voebem.information_schema.columns
WHERE table_schema = 'bronze'
ORDER BY table_name, ordinal_position

In [0]:
%sql
-- Primeiros 5 registros da tabela vra
SELECT *
FROM voebem.bronze.vra
LIMIT 5

In [0]:
%sql
-- Primeiros 5 registros da tabela aerodromos
SELECT * FROM voebem.bronze.aerodromos LIMIT 5

In [0]:
%sql
-- Primeiros 5 registros da tabela empresas_nacionais
SELECT * FROM voebem.bronze.empresas_nacionais LIMIT 5

In [0]:
%sql
-- Primeiros 5 registros da tabela empresas_estrangeiras
SELECT * FROM voebem.bronze.empresas_estrangeiras LIMIT 5

In [0]:
%sql
-- Primeiros 5 registros da tabela codigos_operacao
SELECT * FROM voebem.bronze.codigos_operacao LIMIT 5

## ✅ Respostas do Desafio 1 — Validação da Camada Bronze

Após a execução das células acima, aqui estão as respostas às três perguntas do desafio:

---

### 1. Quantos registros existem na tabela?

A camada Bronze do projeto **voebem** contém **5 tabelas** com um total de **1.016.091 registros**:

| Tabela | Registros |
|---|---|
| `vra` | 1.014.705 |
| `empresas_nacionais` | 729 |
| `aerodromos` | 496 |
| `empresas_estrangeiras` | 148 |
| `codigos_operacao` | 13 |

A tabela principal (`vra`) concentra a maior parte dos dados, com mais de 1 milhão de voos ingeridos a partir de 12 arquivos CSV mensais.

---

### 2. Quais são as colunas disponíveis?

Todas as colunas de dados são do tipo `STRING` — regra da camada Bronze: preservar o dado bruto sem tipagem. A única exceção é `_ingerido_em` (`TIMESTAMP`), adicionada na ingestão para auditoria.

**`vra` (14 colunas):** `icao_empresa`, `numero_voo`, `codigo_di`, `codigo_tipo_linha`, `icao_origem`, `icao_destino`, `partida_prevista`, `partida_real`, `chegada_prevista`, `chegada_real`, `situacao_voo`, `codigo_justificativa`, `_arquivo_origem`, `_ingerido_em`

**`aerodromos` (12 colunas):** `icao`, `ciad`, `nome`, `municipio`, `uf`, `municipio_servido`, `uf_servido`, `latitude`, `longitude`, `altitude`, `situacao`, `_ingerido_em`

**`empresas_nacionais` (9 colunas):** `icao`, `sigla_iata`, `razao_social`, `servico`, `cidade`, `uf`, `situacao`, `_arquivo_origem`, `_ingerido_em`

**`empresas_estrangeiras` (9 colunas):** `icao`, `sigla_iata`, `razao_social`, `servico`, `cidade`, `uf`, `situacao`, `_arquivo_origem`, `_ingerido_em`

**`codigos_operacao` (3 colunas):** `dominio`, `codigo`, `descricao`

---

### 3. Quais são os primeiros registros armazenados?

Os primeiros registros de cada tabela foram exibidos nas células 4 a 8. Exemplos:

- **`vra`**: voos da empresa LPE (voos 2395 e 2402) em janeiro/2026, rota SBFL ↔ SPJC, situados como `REALIZADO`.
- **`aerodromos`**: aeródromos como SBRB (Plácido de Castro / Rio Branco / AC) e SWPI (Parintins / AM), com coordenadas geográficas.
- **`empresas_nacionais`**: empresas aéreas como AGRICENTER AVIAÇÃO AGRÍCOLA LTDA. e AERO AGRÍCOLA TELES PIRES LTDA, situadas como `ATIVA`.
- **`empresas_estrangeiras`**: empresas como WAMOS AIR S.A. (PLM) e SKY TAXI (IGA), operando no Brasil.
- **`codigos_operacao`**: mapeamentos como `codigo_di = 0` → `Etapa Regular`, `codigo_di = 2` → `Etapa Extra`.

---

### Conclusão

A ingestão da camada Bronze foi concluída com sucesso. Todos os dados estão disponíveis, sem modificações, preservando a fidelidade da fonte. As colunas de auditoria (`_arquivo_origem` e `_ingerido_em`) garantem a rastreabilidade de cada registro ingerido.